In [10]:
import numpy as np
import pandas as pd

np.random.seed(50)
n = 1500

df_compras = pd.DataFrame({
    'id_cliente': np.random.randint(10000, 99999, n),
    'monto_compra': np.random.uniform(10.5, 500.0, n).round(2),
    'categoria': np.random.choice(['Electrónica', 'Ropa', 'Hogar', 'Libros'], n),
    'fecha': pd.date_range(start='2026-01-01', periods=n, freq='min')
})
print(f"Dataset inicializado con {len(df_compras)} compras.")

Dataset inicializado con 1500 compras.


## Bining (Segmentación por intervalos)
En estadística es el clásico agrupamiento por intervalos para crear categorías de una variable numérica continua
creamos tres intervalos para monto_compra: Bajo (0, 50), Medio (50, 200), Alto (200, 500). Creamos una nueva variable categórica con estos.

In [11]:
segmentos = [0, 50, 200, 500]
etiquetas = ['Bajo', 'Medio', 'Alto']
df_compras['nivel_gasto'] = pd.cut(df_compras['monto_compra'], bins = segmentos, labels = etiquetas)

## Manipulación de Fechas y de Tiempo

In [12]:
df_compras['hora'] = df_compras['fecha'].dt.hour
df_compras['dia_semana'] = df_compras['fecha'].dt.day_of_week

print(df_compras[['fecha', 'hora', 'dia_semana']].head())

                fecha  hora  dia_semana
0 2026-01-01 00:00:00     0           3
1 2026-01-01 00:01:00     0           3
2 2026-01-01 00:02:00     0           3
3 2026-01-01 00:03:00     0           3
4 2026-01-01 00:04:00     0           3


In [13]:
df_compras['bloque_dia'] = pd.cut(df_compras['hora'], bins = [-1, 6, 12, 20, 24], labels = ['Madrugada', 'Mañana', 'Tarde', 'Noche'])
print(df_compras[['hora', 'bloque_dia']].head())

   hora bloque_dia
0     0  Madrugada
1     0  Madrugada
2     0  Madrugada
3     0  Madrugada
4     0  Madrugada


In [14]:
df_compras['bloque_dia'].value_counts()

bloque_dia
Madrugada    480
Tarde        480
Mañana       360
Noche        180
Name: count, dtype: int64

## Creación de variables condicionales con np.where()
np.where(condición, valor_si_verdadero, valor_si_falso)

In [16]:
#Creación de la condición combinada
condicion_vip = (df_compras['categoria'] == 'Electrónica') & (df_compras['nivel_gasto'] == 'Alto')

#Creamos una nueva variable binaria
df_compras['alerta_vip'] = np.where(condicion_vip, 'Revisar', 'Normal')

print(df_compras['alerta_vip'].value_counts())

alerta_vip
Normal     1279
Revisar     221
Name: count, dtype: int64


## Mapeo y Transformción de de Categorías ( .map() ) 
Permite transformar etiquetas en números utilizando un diccionario

In [17]:
#Definino el diccionario con las relaciones entre etiqueta y valor numérico que queremos establecer
diccionario_prioridad = {'Bajo' : 1, 'Medio' : 2, 'Alto' : 3}

#Creamos una nueva columna a partir de esta equivalencia definida
df_compras['puntos_prioridad'] = df_compras['nivel_gasto'].map(diccionario_prioridad)

#Veamos si se han producido los cambios de la forma correcta

print(df_compras[['puntos_prioridad', 'nivel_gasto']].head())

  puntos_prioridad nivel_gasto
0                3        Alto
1                3        Alto
2                2       Medio
3                1        Bajo
4                3        Alto


## Creación de Variables Dummy o One-Hot Encoding ( pd.get_dummies() )
Este tipo de varables consisten en dividir variables categóricas en otras variables dicotómicas, una por cada
categoría, de modo que se le asigna un 1 a quien pertenece a la categoría que representa la variable o un 0 al registro
que no perteneca a la misma. ESto se hace con el fin de poder tratar estas variables categóricas de forma numérica
para incluirlas en modelos de ML.

In [18]:
#Convertimos la variable categoria en variables dummy
df_transformado = pd.get_dummies(df_compras, columns = ['categoria'])

#Veamos las columnas que se han generado en df_transformado
print(df_transformado.columns)

Index(['id_cliente', 'monto_compra', 'fecha', 'nivel_gasto', 'hora',
       'dia_semana', 'bloque_dia', 'alerta_vip', 'puntos_prioridad',
       'categoria_Electrónica', 'categoria_Hogar', 'categoria_Libros',
       'categoria_Ropa'],
      dtype='object')
